# 05 — Test a New Song

Drop any MP3 or WAV path into the cell below. Get back:
- Which Leiden community it belongs to (k-NN vote confidence)
- Top-N most similar tracks from your library
- Where it sits on the 2D UMAP map (visualization only)

The query is embedded/standardized through the **exact same transform** as the
corpus, with dimension assertions on the way in.

**Requires** the migrated notebook `02` (Phase 1) or `04` (Phase 2) to have been
run, producing `models/pipeline_phase{N}.pkl` (Leiden format), `index_phase{N}.*`,
and `embedding_2d_phase{N}.npy`.

In [ ]:
import sys
sys.path.insert(0, '..')  # optional now that anther_ml is `pip install -e .`

import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from anther_ml.embedding import load_mert, get_embedding
from anther_ml.cluster import load_leiden, assign_cluster_knn
from anther_ml.similarity import SongIndex

print('imports ok')

## ⬇️ Change this cell to your song

In [6]:
# ── DROP YOUR SONG PATH HERE ──────────────────────────────────────────────────
MY_SONG = '/Users/mwilliams/Downloads/sweetest thing v7.mp3'
# ─────────────────────────────────────────────────────────────────────────────

TOP_K = 10   # how many nearest neighbors to return
PHASE = 2    # 1 = pre-computed features, 2 = MERT embeddings

## Load models

In [ ]:
pipeline_path = f'../models/pipeline_phase{PHASE}.pkl'
index_path    = f'../models/index_phase{PHASE}'
umap2d_path   = f'../models/embedding_2d_phase{PHASE}.npy'

# load_leiden returns a dict: labels, scaler, pca, reducer_2d, clustering_space,
# n_neighbors, resolution, metric.
pipe  = load_leiden(pipeline_path)
index = SongIndex.load(index_path)
embedding_2d = np.load(umap2d_path)
labels = pipe['labels']

print(f'Loaded Phase {PHASE} Leiden pipeline — {len(index.metadata)} tracks in index')
print(f'  standardized index: {index.standardize}')
print(f'  index config:       {index.config}')
if pipe.get('reducer_2d') is None:
    print('⚠️  No reducer_2d in pipeline — the 2D placement plot will be skipped.')

In [ ]:
if PHASE == 2:
    model, processor, device = load_mert()
    print(f'MERT on: {device}')
else:
    from anther_ml.features import (
        extract_librosa_features, align_to_corpus, fma_feature_columns,
    )
    model = processor = device = None

## Get embedding for the new song

In [ ]:
print(f'Processing: {Path(MY_SONG).name}')

if PHASE == 2:
    # Match the query embedding to the index's recorded config so the query and
    # corpus are produced identically (Workstream E/G).
    cfg = index.config or {}
    query_vec = get_embedding(
        model, processor, MY_SONG, device,
        layer_aggregation=cfg.get('layer_aggregation', 'mean'),
        normalize=cfg.get('loudness_normalize', True),
    )
else:
    series    = extract_librosa_features(MY_SONG)               # labeled Series (FMA order)
    query_vec = align_to_corpus(series, fma_feature_columns())  # aligned array (asserts, Workstream B)

print(f'Embedding shape: {query_vec.shape}')

## Cluster assignment

In [ ]:
# Assign to a Leiden community by k-NN vote against the labeled corpus, using the
# SAME scale/PCA transform as the corpus. Leiden has no noise bucket, so every
# song gets a community.
cluster_id, confidence = assign_cluster_knn(
    query_vec, pipe['clustering_space'], pipe['labels'],
    scaler=pipe['scaler'], pca=pipe['pca'], k=pipe['n_neighbors'],
)
print(f'Cluster: {cluster_id}  (k-NN vote confidence: {confidence:.2f})')

# Genre composition of the assigned cluster — DISPLAY ONLY (never a decision input)
cluster_members = [m for m in index.metadata if m.get('cluster') == cluster_id]
genre_counts = Counter(m.get('genre', 'unknown') for m in cluster_members)
print(f'  Cluster size: {len(cluster_members)} tracks')
print(f'  Genre composition (display only): {genre_counts.most_common(5)}')

## Nearest neighbors

In [11]:
results = index.query(query_vec, top_k=TOP_K)

print(f'Top {TOP_K} most similar tracks to "{Path(MY_SONG).stem}":')
print(f'{"Rank":>4} | {"Score":>6} | {"Artist":<25} | {"Title":<35} | Genre')
print('-' * 95)
for r in results:
    artist = (r.get('artist') or '')[:24]
    name   = (r.get('name')   or '')[:34]
    genre  = r.get('genre', 'unknown') or 'unknown'
    print(f'{r["rank"]:>4} | {r["score"]:>6.4f} | {artist:<25} | {name:<35} | {genre}')

Top 10 most similar tracks to "sweetest thing v7":
Rank |  Score | Artist                    | Title                               | Genre
-----------------------------------------------------------------------------------------------
   1 | 0.9518 | personal                  | esquina ft blvck sam v5             | unknown
   2 | 0.9406 | personal                  | Miseducation Demo                   | unknown
   3 | 0.9312 | personal                  | all_night_unfinished_8099060227791  | unknown
   4 | 0.9286 | personal                  | colture rough mix 1                 | unknown
   5 | 0.9286 | personal                  | colture rough mix 2                 | unknown
   6 | 0.9224 | personal                  | why-cant-we-official-audio          | unknown
   7 | 0.9210 | personal                  | [FREE] Drake Type Beat 2024 - Hear  | unknown
   8 | 0.9193 | personal                  | 007 bounce                          | unknown
   9 | 0.9170 | personal                  | k

## Plot: where does this song land on the UMAP?

In [ ]:
# Project the new song into the SAME 2D space as the corpus:
#   scale → [pca] → reducer_2d.transform  (reducer_2d was fit on the clustering
#   space, i.e. scaled+PCA embedding — never on raw high-dim UMAP dims).
reducer_2d = pipe.get('reducer_2d')
if reducer_2d is None:
    raise RuntimeError(
        'reducer_2d not found in pipeline. Re-run the Phase notebook to save it, then reload here.'
    )

q = query_vec.reshape(1, -1)
if pipe['scaler'] is not None:
    q = pipe['scaler'].transform(q)
if pipe['pca'] is not None:
    q = pipe['pca'].transform(q)
song_2d = reducer_2d.transform(q)  # (1, 2) — same space as embedding_2d

genre_labels = [m.get('genre', 'unknown') or 'unknown' for m in index.metadata]
unique_genres = sorted(set(genre_labels))
palette = sns.color_palette('tab20', n_colors=len(unique_genres))
g2c = {g: palette[i] for i, g in enumerate(unique_genres)}
colors = [g2c[g] for g in genre_labels]

fig, ax = plt.subplots(figsize=(13, 9))

# Background: all indexed songs
ax.scatter(embedding_2d[:, 0], embedding_2d[:, 1],
           c=colors, s=5, alpha=0.5, linewidths=0, zorder=1)

# Highlight top-10 neighbors
for r in results:
    idx = next((i for i, m in enumerate(index.metadata)
                if m.get('name') == r.get('name') and m.get('artist') == r.get('artist')), None)
    if idx is not None:
        ax.scatter(embedding_2d[idx, 0], embedding_2d[idx, 1],
                   c='orange', s=50, zorder=3, edgecolors='darkorange', linewidths=0.5)

# The new song
ax.scatter(song_2d[0, 0], song_2d[0, 1],
           c='red', s=150, zorder=5, edgecolors='black', linewidths=1.5,
           label=Path(MY_SONG).stem)
ax.annotate(Path(MY_SONG).stem,
            (song_2d[0, 0], song_2d[0, 1]),
            xytext=(8, 8), textcoords='offset points', fontsize=9, fontweight='bold')

handles = [plt.Line2D([0],[0], marker='o', color='w',
                       markerfacecolor=g2c[g], markersize=7, label=g)
           for g in unique_genres]
handles += [
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='orange',
               markersize=8, label='Nearest neighbors'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='red',
               markersize=10, label='Your song'),
]
ax.legend(handles=handles, title='Genre (display only)', bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=7)
ax.set_title(f'Phase {PHASE} — "{Path(MY_SONG).stem}" on the song map (2D UMAP = viz only)')
plt.tight_layout()
plt.savefig(f'../models/test_song_phase{PHASE}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to models/test_song_phase{PHASE}.png')